# Manual Testing: Complete 3-Agent Pipeline with CredibilityFilter

This notebook provides hands-on testing of the expanded pipeline using real API keys:

## Pipeline Flow
1. **QueryOrchestrator** - Parses user query → structured search query + intent
2. **TavilyRetriever** - Executes Tavily two-step process (search → extract)
3. **CredibilityFilter** - Scores and filters results by source credibility
4. **Analysis Tools** - Interactive credibility analysis and filtering

## Credibility Scoring Components
- **Domain Reputation (50%)**: Authority and trustworthiness of source domain
- **Recency Score (30%)**: Content freshness with exponential decay
- **Extractability (20%)**: Quality of data extraction from Tavily

## Requirements
- Real OPENAI_API_KEY and TAVILY_API_KEY in .env file
- Backend dependencies installed

## Setup and Imports

In [1]:
import os
import sys
import asyncio
import json
from datetime import datetime, timezone, timedelta
from pprint import pprint
import pandas as pd
from urllib.parse import urlparse

# Add backend to path
sys.path.append('..')

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

print("✅ Environment loaded")
print(f"OPENAI_API_KEY: {'✅ Set' if os.getenv('OPENAI_API_KEY') else '❌ Missing'}")
print(f"TAVILY_API_KEY: {'✅ Set' if os.getenv('TAVILY_API_KEY') else '❌ Missing'}")

✅ Environment loaded
OPENAI_API_KEY: ✅ Set
TAVILY_API_KEY: ✅ Set


In [2]:
# Import our agents and state management
from app.agents.query_orchestrator_agent import QueryOrchestratorAgent
from app.agents.tavily_retriever_agent import TavilyRetrieverAgent
from app.agents.credibility_filter_agent import CredibilityFilterAgent
from app.agents.state import create_initial_state, get_state_summary
from app.config import settings
from app.extractors.domain_config import get_domain_quality_score

print("✅ Agents imported successfully")
print(f"Environment: {settings.ENVIRONMENT}")
print(f"OpenAI Model: {settings.OPENAI_MODEL}")

✅ Agents imported successfully
Environment: development
OpenAI Model: gpt-4o-mini


## Initialize Agents

In [3]:
# Create agent instances
query_agent = QueryOrchestratorAgent()
tavily_agent = TavilyRetrieverAgent()
credibility_agent = CredibilityFilterAgent()

print("✅ Agents initialized:")
print(f"- {query_agent.name} (OpenAI: {query_agent.llm.model_name})")
print(f"- {tavily_agent.name} (Client: {type(tavily_agent.tavily_client).__name__})")
print(f"- {credibility_agent.name} (Threshold: {credibility_agent.credibility_threshold})")

[Query Orchestrator] Initialized Query Orchestrator with llm_provider: OpenAI and model: gpt-4o-mini
[Tavily Retriever] Initialized with development configuration
[Credibility Filter] Initialized CredibilityFilterAgent
✅ Agents initialized:
- Query Orchestrator (OpenAI: gpt-4o-mini)
- Tavily Retriever (Client: OptimizedTavilyClient)
- Credibility Filter (Threshold: 0.4)


## Test Configuration

In [4]:
# Test Tavily configuration
config_test = await tavily_agent.test_configuration()
print("🔧 Tavily Configuration:")
pprint(config_test)

# Show credibility filter configuration
print("\n🎯 CredibilityFilter Configuration:")
print(f"  Primary threshold: {credibility_agent.credibility_threshold}")
print(f"  Fallback thresholds: {credibility_agent.fallback_thresholds}")
print(f"  Min results fallback: {credibility_agent.min_results_fallback}")
print(f"  Intent weights available: {list(credibility_agent.intent_weights.keys())}")

[Tavily Retriever] Testing Tavily configuration
[Tavily Retriever] Configuration test successful
🔧 Tavily Configuration:
{'agent_name': 'Tavily Retriever',
 'api_key_available': True,
 'client_type': 'OptimizedTavilyClient',
 'config': {'coverage_threshold': 0.6,
            'enable_fallback': False,
            'enable_intent_optimization': True,
            'enable_map_api': False,
            'enable_quality_filter': False,
            'map_max_depth': 1,
            'map_max_results': 10,
            'max_concurrent': 2,
            'max_results': 5,
            'min_domain_quality': 0.5,
            'query_max_length': 400,
            'search_depth': 'basic'}}

🎯 CredibilityFilter Configuration:
  Primary threshold: 0.4
  Fallback thresholds: [0.3, 0.2, 0.1]
  Min results fallback: 3
  Intent weights available: ['product_search', 'review_search', 'comparison', 'general']


## Complete Pipeline Test Functions

In [5]:
async def test_complete_pipeline(raw_query: str, run_id: str = None):
    """
    Complete 3-agent pipeline test: QueryOrchestrator → TavilyRetriever → CredibilityFilter
    """
    if not run_id:
        run_id = f"manual_test_{datetime.now().strftime('%H%M%S')}"
    
    print(f"\n🚀 Testing Complete Pipeline: '{raw_query}'")
    print("=" * 70)
    
    # Create initial state
    state = create_initial_state(raw_query=raw_query, run_id=run_id)
    print(f"📋 Initial state created (run_id: {run_id})")
    
    try:
        # Step 1: QueryOrchestrator
        print("\n1️⃣ QueryOrchestrator Processing...")
        start_time = datetime.now()
        state = await query_agent.process(state)
        query_time = (datetime.now() - start_time).total_seconds()
        
        search_query = state.get("search_query")
        if search_query:
            print(f"✅ Query parsed successfully ({query_time:.2f}s)")
            print(f"   Intent: {search_query.intent}")
            print(f"   Category: {search_query.category}")
            print(f"   Normalized: '{search_query.normalized_query}'")
            if search_query.budget_max:
                print(f"   Budget: ${search_query.budget_max}")
            if search_query.constraints:
                print(f"   Constraints: {search_query.constraints}")
        else:
            print("❌ Query parsing failed")
            return state
        
        # Step 2: TavilyRetriever
        print("\n2️⃣ TavilyRetriever Processing...")
        start_time = datetime.now()
        state = await tavily_agent.process(state)
        tavily_time = (datetime.now() - start_time).total_seconds()
        
        search_results = state.get("raw_search_results", [])
        extracted_content = state.get("extracted_content", [])
        coverage_score = state.get("coverage_score", 0.0)
        
        print(f"✅ Tavily processing completed ({tavily_time:.2f}s)")
        print(f"   Search results: {len(search_results)}")
        print(f"   Extracted content: {len(extracted_content)}")
        print(f"   Coverage score: {coverage_score:.2f}")
        
        # Step 3: CredibilityFilter
        print("\n3️⃣ CredibilityFilter Processing...")
        start_time = datetime.now()
        state = await credibility_agent.process(state)
        credibility_time = (datetime.now() - start_time).total_seconds()
        
        filtered_results = state.get("credibility_filtered_results", [])
        
        print(f"✅ Credibility filtering completed ({credibility_time:.2f}s)")
        print(f"   Filtered results: {len(filtered_results)} (from {len(search_results)})")
        
        if filtered_results:
            scores = [r.get("credibility_score", 0) for r in filtered_results]
            avg_score = sum(scores) / len(scores)
            print(f"   Average credibility: {avg_score:.3f}")
            print(f"   Score range: {min(scores):.3f} - {max(scores):.3f}")
        
        # Show agent execution summary
        summary = get_state_summary(state)
        total_time = query_time + tavily_time + credibility_time
        print(f"\n📊 Pipeline Summary:")
        print(f"   Total time: {total_time:.2f}s")
        print(f"   Agents completed: {summary['progress']['agents_completed']}")
        print(f"   Total cost: ${summary['progress']['total_cost_usd']:.4f}")
        
        return state
        
    except Exception as e:
        print(f"❌ Pipeline error: {e}")
        import traceback
        traceback.print_exc()
        return state

def analyze_credibility_scores(state: dict, detailed: bool = True):
    """
    Analyze credibility scores in detail
    """
    filtered_results = state.get("credibility_filtered_results", [])
    
    if not filtered_results:
        print("No filtered results to analyze")
        return
    
    print(f"\n🔍 Credibility Analysis ({len(filtered_results)} results):")
    print("=" * 60)
    
    for i, result in enumerate(filtered_results, 1):
        url = result.get("url", "No URL")
        domain = urlparse(url).netloc.replace("www.", "")
        title = result.get("title", "No title")
        score = result.get("credibility_score", 0)
        breakdown = result.get("credibility_breakdown", {})
        
        print(f"\n{i}. {title[:60]}...")
        print(f"   Domain: {domain}")
        print(f"   Overall Score: {score:.3f}")
        
        if detailed and breakdown:
            print(f"   📊 Breakdown:")
            print(f"      Domain Score: {breakdown.get('domain_score', 0):.3f}")
            print(f"      Recency Score: {breakdown.get('recency_score', 0):.3f}")
            print(f"      Extractability: {breakdown.get('extractability_score', 0):.3f}")
            print(f"      Intent: {breakdown.get('intent', 'unknown')}")
        
        # Show warnings if any
        if "credibility_warning" in result:
            print(f"   ⚠️ Warning: {result['credibility_warning']}")

def compare_before_after_filtering(state: dict):
    """
    Compare results before and after credibility filtering
    """
    raw_results = state.get("raw_search_results", [])
    filtered_results = state.get("credibility_filtered_results", [])
    
    print(f"\n📊 Before/After Filtering Comparison:")
    print("=" * 50)
    print(f"Raw results: {len(raw_results)}")
    print(f"Filtered results: {len(filtered_results)}")
    print(f"Filtering ratio: {len(filtered_results)/max(len(raw_results), 1)*100:.1f}%")
    
    if filtered_results:
        # Show which domains made it through
        filtered_domains = [urlparse(r.get("url", "")).netloc.replace("www.", "") for r in filtered_results]
        domain_counts = {}
        for domain in filtered_domains:
            domain_counts[domain] = domain_counts.get(domain, 0) + 1
        
        print(f"\n🏆 Domains that passed filtering:")
        for domain, count in sorted(domain_counts.items(), key=lambda x: x[1], reverse=True):
            domain_score = get_domain_quality_score(domain)
            print(f"   {domain}: {count} results (domain score: {domain_score:.3f})")

def show_filtering_thresholds_impact(state: dict):
    """
    Show how different filtering thresholds would impact results
    """
    filtered_results = state.get("credibility_filtered_results", [])
    
    if not filtered_results:
        return
    
    scores = [r.get("credibility_score", 0) for r in filtered_results]
    thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
    
    print(f"\n🎯 Impact of Different Filtering Thresholds:")
    print("=" * 50)
    
    for threshold in thresholds:
        passing = sum(1 for score in scores if score >= threshold)
        percentage = passing / len(scores) * 100
        indicator = "🟢" if threshold <= 0.4 else "🟡" if threshold <= 0.6 else "🔴"
        current = " (CURRENT)" if threshold == 0.4 else ""
        print(f"   {indicator} Threshold {threshold:.1f}: {passing}/{len(scores)} results ({percentage:.1f}%){current}")

## Interactive Testing

Now you can test different queries manually and analyze the credibility filtering results!

### Test 1: Product Search - Wireless Earbuds

In [6]:
# Test product search with budget constraint
result_state = await test_complete_pipeline("best wireless earbuds under $100")


🚀 Testing Complete Pipeline: 'best wireless earbuds under $100'
📋 Initial state created (run_id: manual_test_004809)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'best wireless earbuds under $100'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "best wireless earbuds under $100",
  "normalized_query": "best wireless earbuds",
  "intent": "product_search",
  "category": "headphones",
  "brand": null,
  "budget_min": null,
  "budget_max": 100.0,
  "constraints": [
    "wireless"
  ],
  "priorities": [
    "price"
  ],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "best wireless earbuds",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (6.66s)
   Intent: product_search
   Category: headphones
   Normalized: 'best wireless earbuds'
   Budget: $100.0
   Constraints: ['wireless']

2️⃣ TavilyRetriever Processing...
[Tavily Retriever] 

In [7]:
# Analyze the credibility scores in detail
analyze_credibility_scores(result_state, detailed=True)


🔍 Credibility Analysis (5 results):

1. DEWALT Heavy Duty True Wireless Ear Buds, Bluetooth ......
   Domain: amazon.com
   Overall Score: 0.790
   📊 Breakdown:
      Domain Score: 1.000
      Recency Score: 0.700
      Extractability: 0.100
      Intent: product_search

2. Wireless Earbuds - Headphones...
   Domain: bestbuy.com
   Overall Score: 0.760
   📊 Breakdown:
      Domain Score: 0.950
      Recency Score: 0.700
      Extractability: 0.100
      Intent: product_search

3. OnePlus Buds 4 True Wireless Earbuds – Noise ......
   Domain: bestbuy.com
   Overall Score: 0.760
   📊 Breakdown:
      Domain Score: 0.950
      Recency Score: 0.700
      Extractability: 0.100
      Intent: product_search

4. Smart Translation Wireless Earbuds with 100+ Languages ......
   Domain: bestbuy.com
   Overall Score: 0.760
   📊 Breakdown:
      Domain Score: 0.950
      Recency Score: 0.700
      Extractability: 0.100
      Intent: product_search

5. Earbuds and In-Ear Headphones...
   Domain: wa

In [8]:
# Compare before/after filtering
compare_before_after_filtering(result_state)


📊 Before/After Filtering Comparison:
Raw results: 5
Filtered results: 5
Filtering ratio: 100.0%

🏆 Domains that passed filtering:
   bestbuy.com: 3 results (domain score: 0.950)
   amazon.com: 1 results (domain score: 1.000)
   walmart.com: 1 results (domain score: 0.900)


In [9]:
# Show impact of different thresholds
show_filtering_thresholds_impact(result_state)


🎯 Impact of Different Filtering Thresholds:
   🟢 Threshold 0.2: 5/5 results (100.0%)
   🟢 Threshold 0.3: 5/5 results (100.0%)
   🟢 Threshold 0.4: 5/5 results (100.0%) (CURRENT)
   🟡 Threshold 0.5: 5/5 results (100.0%)
   🟡 Threshold 0.6: 5/5 results (100.0%)
   🔴 Threshold 0.7: 5/5 results (100.0%)
   🔴 Threshold 0.8: 0/5 results (0.0%)


### Test 2: Review Search - Product Reviews

In [10]:
# Test review search (different intent weighting)
review_state = await test_complete_pipeline("MacBook Pro M4 review 2025")


🚀 Testing Complete Pipeline: 'MacBook Pro M4 review 2025'
📋 Initial state created (run_id: manual_test_004818)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'MacBook Pro M4 review 2025'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "MacBook Pro M4 review 2025",
  "normalized_query": "MacBook Pro M4 2025",
  "intent": "review_search",
  "category": "laptop",
  "brand": "apple",
  "budget_min": null,
  "budget_max": null,
  "constraints": [
    "2025"
  ],
  "priorities": [],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "MacBook Pro M4 2025",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (1.61s)
   Intent: review_search
   Category: laptop
   Normalized: 'MacBook Pro M4 2025'
   Constraints: ['2025']

2️⃣ TavilyRetriever Processing...
[Tavily Retriever] Starting Tavily search and extraction
[Tavily Retriever] Processing query

In [11]:
# Analyze review search credibility
analyze_credibility_scores(review_state, detailed=True)
compare_before_after_filtering(review_state)


🔍 Credibility Analysis (5 results):

1. Best MacBooks We've Tested (September 2025)...
   Domain: cnet.com
   Overall Score: 0.670
   📊 Breakdown:
      Domain Score: 0.900
      Recency Score: 0.700
      Extractability: 0.100
      Intent: review_search

2. The Best Apple MacBook Air and MacBook Pro Laptops for ......
   Domain: pcmag.com
   Overall Score: 0.625
   📊 Breakdown:
      Domain Score: 0.800
      Recency Score: 0.700
      Extractability: 0.100
      Intent: review_search

3. Apple MacBook Pro 14-Inch (2024, M4) Review...
   Domain: pcmag.com
   Overall Score: 0.625
   📊 Breakdown:
      Domain Score: 0.800
      Recency Score: 0.700
      Extractability: 0.100
      Intent: review_search

4. MacBook Air (M4, 2025) review: Blue skies ahead...
   Domain: tomshardware.com
   Overall Score: 0.580
   📊 Breakdown:
      Domain Score: 0.700
      Recency Score: 0.700
      Extractability: 0.100
      Intent: review_search

5. MacBook Air vs. MacBook Pro 2025: Which Mac should

### Test 3: Comparison Search - Product Comparison

In [12]:
# Test comparison search (emphasizes extractability)
comparison_state = await test_complete_pipeline("iPhone 15 vs Samsung Galaxy S24 camera quality")


🚀 Testing Complete Pipeline: 'iPhone 15 vs Samsung Galaxy S24 camera quality'
📋 Initial state created (run_id: manual_test_004823)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'iPhone 15 vs Samsung Galaxy S24 camera quality'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "iPhone 15 vs Samsung Galaxy S24 camera quality",
  "normalized_query": "iPhone 15 Samsung Galaxy S24 camera quality",
  "intent": "comparison",
  "category": "smartphone",
  "brand": null,
  "budget_min": null,
  "budget_max": null,
  "constraints": [
    "camera quality"
  ],
  "priorities": [
    "camera"
  ],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "iPhone 15 Samsung Galaxy S24 camera quality",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (1.32s)
   Intent: comparison
   Category: smartphone
   Normalized: 'iPhone 15 Samsung Galaxy S24 camera qual

In [13]:
# Analyze comparison search credibility
analyze_credibility_scores(comparison_state, detailed=True)
show_filtering_thresholds_impact(comparison_state)


🔍 Credibility Analysis (5 results):

1. Compare - Samsung Galaxy S24...
   Domain: gsmarena.com
   Overall Score: 0.490
   📊 Breakdown:
      Domain Score: 0.700
      Recency Score: 0.700
      Extractability: 0.100
      Intent: comparison

2. Samsung Galaxy S24 Plus vs Samsung Galaxy Z Fold 7...
   Domain: versus.com
   Overall Score: 0.490
   📊 Breakdown:
      Domain Score: 0.700
      Recency Score: 0.700
      Extractability: 0.100
      Intent: comparison

3. Samsung Galaxy S24 FE...
   Domain: notebookcheck.net
   Overall Score: 0.490
   📊 Breakdown:
      Domain Score: 0.700
      Recency Score: 0.700
      Extractability: 0.100
      Intent: comparison

4. Samsung Galaxy S24 Ultra...
   Domain: notebookcheck.net
   Overall Score: 0.490
   📊 Breakdown:
      Domain Score: 0.700
      Recency Score: 0.700
      Extractability: 0.100
      Intent: comparison

5. Apple iPhone 17 Pro Max vs Samsung Galaxy S25 Ultra...
   Domain: versus.com
   Overall Score: 0.490
   📊 Breakdown:

### Custom Query Testing

Use this cell to test your own queries and analyze the credibility filtering:

In [14]:
# Your custom query here
custom_query = "gaming monitor 4K 144Hz under $500"
custom_result = await test_complete_pipeline(custom_query)


🚀 Testing Complete Pipeline: 'gaming monitor 4K 144Hz under $500'
📋 Initial state created (run_id: manual_test_004827)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'gaming monitor 4K 144Hz under $500'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "gaming monitor 4K 144Hz under $500",
  "normalized_query": "gaming monitor 4K 144Hz",
  "intent": "product_search",
  "category": "monitor",
  "brand": null,
  "budget_min": null,
  "budget_max": 500.0,
  "constraints": [
    "4K",
    "144Hz",
    "gaming"
  ],
  "priorities": [
    "performance"
  ],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "gaming monitor 4K 144Hz",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (1.76s)
   Intent: product_search
   Category: monitor
   Normalized: 'gaming monitor 4K 144Hz'
   Budget: $500.0
   Constraints: ['4K', '144Hz', 'gaming']

2️⃣ Tav

In [15]:
# Analyze your custom results
analyze_credibility_scores(custom_result, detailed=True)
compare_before_after_filtering(custom_result)
show_filtering_thresholds_impact(custom_result)


🔍 Credibility Analysis (5 results):

1. Gaming Monitors...
   Domain: bestbuy.com
   Overall Score: 0.760
   📊 Breakdown:
      Domain Score: 0.950
      Recency Score: 0.700
      Extractability: 0.100
      Intent: product_search

2. Spectrum One 27-In. 4K HDR 144-Hz Monitor with USB-C ......
   Domain: bestbuy.com
   Overall Score: 0.760
   📊 Breakdown:
      Domain Score: 0.950
      Recency Score: 0.700
      Extractability: 0.100
      Intent: product_search

3. 144hz monitor...
   Domain: bestbuy.com
   Overall Score: 0.760
   📊 Breakdown:
      Domain Score: 0.950
      Recency Score: 0.700
      Extractability: 0.100
      Intent: product_search

4. Samsung 27" Odyssey G70D 4K UHD IPS 144Hz 1ms Fast ......
   Domain: target.com
   Overall Score: 0.730
   📊 Breakdown:
      Domain Score: 0.900
      Recency Score: 0.700
      Extractability: 0.100
      Intent: product_search

5. LG UltraGear 32G810SA-W 31.5" 4K HDR 144 Hz Smart ......
   Domain: bhphotovideo.com
   Overall Sc

## Advanced Analysis: Intent-Aware Scoring

In [16]:
def compare_intent_scoring():
    """
    Show how different intents affect credibility weighting
    """
    print("🎯 Intent-Aware Scoring Weights:")
    print("=" * 40)
    
    for intent, weights in credibility_agent.intent_weights.items():
        print(f"\n{intent.upper()}:")
        print(f"   Domain: {weights['domain']*100:.0f}%")
        print(f"   Recency: {weights['recency']*100:.0f}%")
        print(f"   Extractability: {weights['extractability']*100:.0f}%")
        
        # Explain the rationale
        if intent == "product_search":
            print(f"   💡 Emphasizes domain authority (e-commerce trust)")
        elif intent == "review_search":
            print(f"   💡 Balanced approach (fresh reviews from trusted sources)")
        elif intent == "comparison":
            print(f"   💡 Emphasizes extractability (structured comparison data)")

compare_intent_scoring()

🎯 Intent-Aware Scoring Weights:

PRODUCT_SEARCH:
   Domain: 60%
   Recency: 25%
   Extractability: 15%
   💡 Emphasizes domain authority (e-commerce trust)

REVIEW_SEARCH:
   Domain: 45%
   Recency: 35%
   Extractability: 20%
   💡 Balanced approach (fresh reviews from trusted sources)

COMPARISON:
   Domain: 40%
   Recency: 25%
   Extractability: 35%
   💡 Emphasizes extractability (structured comparison data)

GENERAL:
   Domain: 50%
   Recency: 30%
   Extractability: 20%


## Performance Analysis

In [17]:
# Performance test with multiple queries
performance_queries = [
    "laptop for programming",
    "smartphone camera review", 
    "tablet vs laptop comparison",
    "wireless headphones noise canceling"
]

performance_results = []

print("⚡ Performance Testing:")
print("=" * 30)

for i, query in enumerate(performance_queries, 1):
    print(f"\n{i}. Testing: '{query}'")
    start_time = datetime.now()
    
    result = await test_complete_pipeline(query, f"perf_test_{i}")
    
    total_time = (datetime.now() - start_time).total_seconds()
    summary = get_state_summary(result)
    
    filtered_count = len(result.get("credibility_filtered_results", []))
    raw_count = len(result.get("raw_search_results", []))
    
    performance_results.append({
        "query": query,
        "time_seconds": total_time,
        "cost_usd": summary['progress']['total_cost_usd'],
        "raw_results": raw_count,
        "filtered_results": filtered_count,
        "filter_ratio": filtered_count / max(raw_count, 1)
    })

print("\n📊 Performance Summary:")
print("=" * 50)
df = pd.DataFrame(performance_results)
print(df.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

print(f"\n🎯 Performance Metrics:")
print(f"   Average time: {df['time_seconds'].mean():.1f}s")
print(f"   Average cost: ${df['cost_usd'].mean():.4f}")
print(f"   Average filter ratio: {df['filter_ratio'].mean():.1%}")

⚡ Performance Testing:

1. Testing: 'laptop for programming'

🚀 Testing Complete Pipeline: 'laptop for programming'
📋 Initial state created (run_id: perf_test_1)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'laptop for programming'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "laptop for programming",
  "normalized_query": "laptop for programming",
  "intent": "product_search",
  "category": "laptop",
  "brand": null,
  "budget_min": null,
  "budget_max": null,
  "constraints": [],
  "priorities": [
    "performance"
  ],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "laptop for programming",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (1.87s)
   Intent: product_search
   Category: laptop
   Normalized: 'laptop for programming'

2️⃣ TavilyRetriever Processing...
[Tavily Retriever] Starting Tavily search and extraction
[Ta

## Deep Dive: Domain Quality Analysis

In [18]:
# Analyze domain quality scores for common domains
common_domains = [
    "amazon.com", "bestbuy.com", "walmart.com", "target.com",
    "wirecutter.nytimes.com", "cnet.com", "techradar.com",
    "pcmag.com", "tomsguide.com", "digitaltrends.com",
    "reddit.com", "quora.com", "unknown-site.com"
]

print("🏆 Domain Quality Scores:")
print("=" * 40)

domain_scores = []
for domain in common_domains:
    score = get_domain_quality_score(domain)
    domain_scores.append({"domain": domain, "score": score})
    
    # Visual indicator
    if score >= 0.9:
        indicator = "🟢 Premium"
    elif score >= 0.8:
        indicator = "🟡 High"
    elif score >= 0.7:
        indicator = "🟠 Medium"
    else:
        indicator = "🔴 Low"
        
    print(f"   {indicator:12} {domain:25} {score:.3f}")

# Show distribution
df_domains = pd.DataFrame(domain_scores)
print(f"\n📊 Domain Score Distribution:")
print(f"   Mean: {df_domains['score'].mean():.3f}")
print(f"   Std: {df_domains['score'].std():.3f}")
print(f"   Range: {df_domains['score'].min():.3f} - {df_domains['score'].max():.3f}")

🏆 Domain Quality Scores:
   🟢 Premium    amazon.com                1.000
   🟢 Premium    bestbuy.com               0.950
   🟢 Premium    walmart.com               0.900
   🟢 Premium    target.com                0.900
   🟢 Premium    wirecutter.nytimes.com    0.950
   🟢 Premium    cnet.com                  0.900
   🟡 High       techradar.com             0.850
   🟡 High       pcmag.com                 0.800
   🟠 Medium     tomsguide.com             0.700
   🟠 Medium     digitaltrends.com         0.750
   🔴 Low        reddit.com                0.600
   🔴 Low        quora.com                 0.500
   🟠 Medium     unknown-site.com          0.700

📊 Domain Score Distribution:
   Mean: 0.808
   Std: 0.150
   Range: 0.500 - 1.000


## Experimental: Custom Threshold Testing

In [19]:
def test_custom_threshold(state: dict, custom_threshold: float):
    """
    Test filtering with a custom threshold
    """
    filtered_results = state.get("credibility_filtered_results", [])
    
    if not filtered_results:
        print("No results to test with custom threshold")
        return
    
    # Apply custom threshold
    custom_filtered = [r for r in filtered_results if r.get("credibility_score", 0) >= custom_threshold]
    
    print(f"\n🎯 Custom Threshold Testing (≥{custom_threshold}):")
    print(f"   Original results: {len(filtered_results)}")
    print(f"   Custom filtered: {len(custom_filtered)}")
    print(f"   Retention rate: {len(custom_filtered)/len(filtered_results)*100:.1f}%")
    
    if custom_filtered:
        avg_score = sum(r.get("credibility_score", 0) for r in custom_filtered) / len(custom_filtered)
        print(f"   Average score: {avg_score:.3f}")
        
        print(f"\n🏆 Results passing custom threshold:")
        for i, result in enumerate(custom_filtered[:3], 1):
            domain = urlparse(result.get("url", "")).netloc.replace("www.", "")
            score = result.get("credibility_score", 0)
            title = result.get("title", "No title")[:50]
            print(f"   {i}. {domain} ({score:.3f}) - {title}...")

# Test with different thresholds
if 'custom_result' in locals():
    test_custom_threshold(custom_result, 0.6)
    test_custom_threshold(custom_result, 0.8)
else:
    print("Run a custom query first to test custom thresholds")


🎯 Custom Threshold Testing (≥0.6):
   Original results: 5
   Custom filtered: 5
   Retention rate: 100.0%
   Average score: 0.724

🏆 Results passing custom threshold:
   1. bestbuy.com (0.760) - Gaming Monitors...
   2. bestbuy.com (0.760) - Spectrum One 27-In. 4K HDR 144-Hz Monitor with USB...
   3. bestbuy.com (0.760) - 144hz monitor...

🎯 Custom Threshold Testing (≥0.8):
   Original results: 5
   Custom filtered: 0
   Retention rate: 0.0%


## Final Summary & Next Steps

🎉 **Congratulations!** You've successfully tested the complete 3-agent pipeline:

### ✅ **What We've Tested:**
1. **QueryOrchestrator** - Intent recognition and query parsing
2. **TavilyRetriever** - Real web search and content extraction
3. **CredibilityFilter** - Multi-factor credibility scoring and filtering

### 🎯 **Key Findings:**
- **Intent-aware scoring** adapts weights based on search type
- **Domain reputation** strongly influences filtering decisions
- **Recency scoring** favors fresh content with exponential decay
- **Extractability** rewards structured, information-rich content
- **Progressive fallback** ensures results even when quality is low

### ➡️ **Next Agent: SpecExtractorAgent**
The pipeline is ready for the next component that will:
- Extract structured product data from credibility-filtered results
- Use Tavily extraction + LLM enhancement for missing fields
- Validate against product schemas
- Calculate field coverage scores

The foundation is solid and ready for structured data extraction! 🚀